# Retraining Gate Gap Handoff V1

This notebook consolidates the current retention gate, SOXL backtest reference baseline, current best replay model floor, and prior model-selection screens into a concrete handoff for the next retraining run. It is not a promotion artifact and does not retain a new actor.

In [1]:
from __future__ import annotations

import json
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / 'AGENTS.md').exists() and (parent / 'SPEC.md').exists():
        ROOT = parent
        break
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'retraining_gate_gap_handoff_v1'
CARD.mkdir(parents=True, exist_ok=True)

def load_json(path: str) -> dict:
    return json.loads((ROOT / path).read_text(encoding='utf-8'))

def as_float(value, default=0.0):
    if value is None or value == '':
        return default
    return float(value)

SOURCES = {
    'retention_gate': 'runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json',
    'retention_gate_csv': 'runtime/model_selection/card/model_retention_walkforward_gate_v1/retention_gate.csv',
    'checkpoint_drift_selector': 'runtime/model_selection/card/cap10_joint_contract_checkpoint_drift_selector_v1/summary.json',
    'checkpoint_selection_csv': 'runtime/model_selection/card/cap10_joint_contract_checkpoint_drift_selector_v1/checkpoint_selection_by_fold.csv',
    'checkpoint_candidates_csv': 'runtime/model_selection/card/cap10_joint_contract_checkpoint_drift_selector_v1/checkpoint_candidate_metrics.csv',
    'bc_ppo_drift_screen': 'runtime/model_selection/card/cap10_joint_contract_bc_ppo_drift_screen_v1/summary.json',
    'bc_slice_screen': 'runtime/model_selection/card/cap10_joint_contract_bc_slice_screen_v1/summary.json',
    'rebound_weighted_bc': 'runtime/model_selection/card/state_objective_rebound_weighted_bc_v1/summary.json',
    'rolling_heuristic_selector': 'runtime/model_selection/card/rolling_heuristic_selector_v1/summary.json',
    'heuristic_strategy_search': 'runtime/model_selection/card/heuristic_strategy_search_v1/summary.json',
}


In [2]:
retention = load_json(SOURCES['retention_gate'])
retention_rows = pd.read_csv(ROOT / SOURCES['retention_gate_csv'])
summary = retention['summary']

profile_rows = retention_rows[retention_rows['profile'] != 'soxl_backtest_policy_reimplementation_v1'].copy()
failure_counts = Counter()
for failures in profile_rows['failures'].fillna(''):
    for failure in str(failures).split(';'):
        if failure:
            failure_counts[failure] += 1

# After strict retention pruning, the CSV may contain only the reference
# baseline. Keep the failure sidecar informative by falling back to the
# protected registry audit rather than writing an empty file.
if not failure_counts:
    for row in retention.get('registry_keep_actor_audit', []):
        for failure in row.get('failures', []):
            if failure:
                failure_counts[failure] += 1

failure_summary = pd.DataFrame(
    [{'failure': key, 'count': value} for key, value in failure_counts.most_common()],
    columns=['failure', 'count'],
)
closest_by_wf_dd = (
    profile_rows.dropna(subset=['worst_wf_max_drawdown_pct'])
    .sort_values(['worst_wf_max_drawdown_pct', 'ytd_return_pct'], ascending=[True, False])
    .head(5)
)
closest_by_ytd = (
    profile_rows.sort_values(['ytd_return_pct', 'worst_wf_max_drawdown_pct'], ascending=[False, True])
    .head(5)
)

failure_summary.to_csv(CARD / 'retention_failure_summary.csv', index=False)
closest_by_wf_dd.to_csv(CARD / 'closest_by_walkforward_drawdown.csv', index=False)
closest_by_ytd.to_csv(CARD / 'closest_by_ytd_return.csv', index=False)

print('passing_profiles', summary['passing_profiles'])
print('top_failures', failure_summary.head(8).to_dict(orient='records'))
print('closest_by_wf_dd', closest_by_wf_dd[['profile','worst_wf_max_drawdown_pct','ytd_return_pct','failures']].to_dict(orient='records'))


passing_profiles []
top_failures [{'failure': 'missing_replay_evidence', 'count': 7}]
closest_by_wf_dd []


In [3]:
checkpoint = load_json(SOURCES['checkpoint_drift_selector'])
bc_ppo = load_json(SOURCES['bc_ppo_drift_screen'])
bc_slice = load_json(SOURCES['bc_slice_screen'])
rebound = load_json(SOURCES['rebound_weighted_bc'])
rolling = load_json(SOURCES['rolling_heuristic_selector'])
heuristic = load_json(SOURCES['heuristic_strategy_search'])

checkpoint_selection = pd.read_csv(ROOT / SOURCES['checkpoint_selection_csv'])
checkpoint_candidates = pd.read_csv(ROOT / SOURCES['checkpoint_candidates_csv'])

source_cards = pd.DataFrame([
    {
        'source': 'model_retention_walkforward_gate_v1',
        'decision': 'no_replayable_profile_passes_retention_gate',
        'promotion_allowed': False,
        'use_for_next_training': True,
        'key_evidence': f"0/{summary['profiles_scanned']} replayable profiles pass; required YTD floor {summary['required_best_return_pct']:.6f}% and MaxDD < {summary['max_walkforward_drawdown_pct']:.2f}% across {','.join(summary['required_wf_windows'])}",
    },
    {
        'source': 'cap10_joint_contract_checkpoint_drift_selector_v1',
        'decision': checkpoint['selection']['decision'],
        'promotion_allowed': False,
        'use_for_next_training': True,
        'key_evidence': '; '.join(checkpoint['conclusion']),
    },
    {
        'source': 'cap10_joint_contract_bc_slice_screen_v1',
        'decision': bc_slice['decision'],
        'promotion_allowed': bool(bc_slice['promotion_allowed']),
        'use_for_next_training': bool(bc_slice.get('ppo_screen_allowed_next', False)),
        'key_evidence': f"selected_config={bc_slice['selected_config']}; min_fold_accuracy={bc_slice['selected_min_fold_accuracy']:.3f}; next={bc_slice['next_mechanism']}",
    },
    {
        'source': 'cap10_joint_contract_bc_ppo_drift_screen_v1',
        'decision': bc_ppo['decision'],
        'promotion_allowed': False,
        'use_for_next_training': False,
        'key_evidence': '; '.join(bc_ppo['conclusion']),
    },
    {
        'source': 'state_objective_rebound_weighted_bc_v1',
        'decision': rebound['selection']['decision'],
        'promotion_allowed': bool(rebound['selection']['promotion_allowed']),
        'use_for_next_training': False,
        'key_evidence': '; '.join(rebound['conclusion']),
    },
    {
        'source': 'rolling_heuristic_selector_v1',
        'decision': rolling['decision'],
        'promotion_allowed': bool(rolling['promotion_allowed']),
        'use_for_next_training': False,
        'key_evidence': rolling['next_action'],
    },
])
source_cards.to_csv(CARD / 'source_card_decisions.csv', index=False)

checkpoint_selection.to_csv(CARD / 'selected_checkpoint_by_fold.csv', index=False)

print(source_cards[['source','decision','use_for_next_training']].to_string(index=False))
print(checkpoint_selection.to_string(index=False))


                                           source                                                               decision  use_for_next_training
              model_retention_walkforward_gate_v1                            no_replayable_profile_passes_retention_gate                   True
cap10_joint_contract_checkpoint_drift_selector_v1     checkpoint_drift_selector_partial_pass_wf2022_blocked_no_promotion                   True
          cap10_joint_contract_bc_slice_screen_v1              joint_contract_bc_slice_ready_for_ppo_screen_no_promotion                   True
      cap10_joint_contract_bc_ppo_drift_screen_v1 joint_contract_bc_initialization_useful_ppo_not_improving_no_promotion                  False
           state_objective_rebound_weighted_bc_v1                          rebound_weighting_no_improvement_no_promotion                  False
                    rolling_heuristic_selector_v1                                   hard_gate_no_rolling_selector_passes                

In [4]:
wf2022 = checkpoint_selection[checkpoint_selection['fold'].eq('wf_2022')].iloc[0].to_dict()
wf2023 = checkpoint_selection[checkpoint_selection['fold'].eq('wf_2023_2024')].iloc[0].to_dict()
ppo_wf2022 = checkpoint_candidates[
    checkpoint_candidates['fold'].eq('wf_2022') & checkpoint_candidates['candidate'].eq('ppo_selected_best')
].iloc[0].to_dict()

handoff = {
    'experiment': 'retraining_gate_gap_handoff_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': 'run_full_walkforward_cap10_joint_contract_bc_checkpoint_training',
    'promotion_allowed': False,
    'next_training_allowed': True,
    'canonical_reference_baseline': summary['reference_profile'],
    'current_best_ytd_return_pct': summary['current_best_return_pct'],
    'required_best_return_pct': summary['required_best_return_pct'],
    'required_soxlbacktest_return_pct': summary['reference_return_pct'],
    'retention_gate': {
        'max_walkforward_drawdown_pct': summary['max_walkforward_drawdown_pct'],
        'required_wf_windows': summary['required_wf_windows'],
        'passing_profile_count': summary['passing_profile_count'],
        'profiles_scanned': summary['profiles_scanned'],
    },
    'selected_training_source': 'cap10_joint_contract_checkpoint_drift_selector_v1',
    'start_from': {
        'candidate_family': 'cap10_joint_contract_bc_initialization_or_ppo_iter0_bc_equivalent',
        'wf_2022_selected_candidate': wf2022['selected_candidate'],
        'wf_2022_selected_relative_return': as_float(wf2022['selected_relative_return']),
        'wf_2022_selected_max_dd': as_float(wf2022['selected_max_dd']),
        'wf_2023_2024_selected_candidate': wf2023['selected_candidate'],
        'wf_2023_2024_selected_relative_return': as_float(wf2023['selected_relative_return']),
        'wf_2023_2024_selected_max_dd': as_float(wf2023['selected_max_dd']),
    },
    'do_not_repeat': [
        'do_not_promote_any_replayable_model_without_complete_wf_2020_wf_2022_wf_2023_2024_metrics',
        'do_not_continue_static_heuristic_or_rolling_selector_branch_without_model_replay_or_bc_candidate_paths',
        'do_not_lengthen_ppo_when_wf_2022_dd_delta_vs_bc_exceeds_2pp',
        'do_not_retry_rebound_weight_only_bc_without_target_enrichment_or_sampling_change',
    ],
    'required_next_run_controls': [
        'add_wf_2020_coverage_before_actor_retention',
        'use_bc_initialization_or_iter0_bc_equivalent_as_checkpoint_reference',
        'apply_checkpoint_drift_stop: ppo_vs_bc_max_dd_delta_pp <= 2.0',
        'apply_checkpoint_drift_stop: ppo_vs_bc_relative_delta_pp >= -5.0',
        'require_fold_gate_before_full_training_continuation',
        'export_daily_paths_for_dashboard_and_retention_gate_review',
    ],
    'nearest_known_gap': {
        'wf_2022_selected_max_dd_gap_to_15pct': round(max(0.0, as_float(wf2022['selected_max_dd']) - 15.0), 6),
        'wf_2022_selected_max_dd_gap_to_20pct': round(max(0.0, as_float(wf2022['selected_max_dd']) - 20.0), 6),
        'wf_2022_ppo_selected_max_dd_delta_vs_bc': round(as_float(ppo_wf2022['max_dd_delta_vs_bc']), 6),
        'wf_2022_ppo_selected_relative_delta_vs_bc': round(as_float(ppo_wf2022['relative_delta_vs_bc']), 6),
        'wf_2023_2024_selected_gate': bool(wf2023['selected_candidate_gate']),
    },
    'acceptance_criteria_for_retention': [
        'all required walk-forward windows present',
        'worst walk-forward MaxDD strictly below 20%',
        'YTD return >= soxl_backtest_policy_reimplementation_v1 YTD return',
        'YTD return >= 95% of current best replay model YTD return',
        'dashboard replay CSV includes date, strategy_nav, position, action, price/close fields',
    ],
    'sidecars': [
        'summary.json',
        'source_card_decisions.csv',
        'retention_failure_summary.csv',
        'closest_by_walkforward_drawdown.csv',
        'closest_by_ytd_return.csv',
        'selected_checkpoint_by_fold.csv',
        'training_objective_handoff.json',
        'conclusion.md',
    ],
}

(CARD / 'training_objective_handoff.json').write_text(json.dumps(handoff, indent=2, sort_keys=True) + '\n', encoding='utf-8')
summary_out = {
    'experiment': handoff['experiment'],
    'created_at': handoff['created_at'],
    'decision': handoff['decision'],
    'promotion_allowed': handoff['promotion_allowed'],
    'next_training_allowed': handoff['next_training_allowed'],
    'retention_gate': handoff['retention_gate'],
    'start_from': handoff['start_from'],
    'nearest_known_gap': handoff['nearest_known_gap'],
    'do_not_repeat': handoff['do_not_repeat'],
    'required_next_run_controls': handoff['required_next_run_controls'],
    'sidecars': handoff['sidecars'],
}
(CARD / 'summary.json').write_text(json.dumps(summary_out, indent=2, sort_keys=True) + '\n', encoding='utf-8')

lines = [
    '# Retraining Gate Gap Handoff V1',
    '',
    f"- Decision: `{handoff['decision']}`.",
    '- Promotion: blocked; this card does not retain or promote a new actor.',
    f"- Retention gate: 0/{summary['profiles_scanned']} replayable profiles pass complete WF + MaxDD/return alignment.",
    f"- SOXLBacktest reference: `{summary['reference_profile']}` YTD `{summary['reference_return_pct']:.6f}%`; current-best alignment floor `{summary['required_best_return_pct']:.6f}%`.",
    f"- Best available next source: cap10 joint-contract BC/checkpoint path. wf_2022 selected MaxDD `{handoff['start_from']['wf_2022_selected_max_dd']:.6f}%`, rel `{handoff['start_from']['wf_2022_selected_relative_return']:.6f}pp`; wf_2023_2024 selected MaxDD `{handoff['start_from']['wf_2023_2024_selected_max_dd']:.6f}%`, rel `{handoff['start_from']['wf_2023_2024_selected_relative_return']:.6f}pp`.",
    f"- Critical stop rule: do not lengthen PPO when wf_2022 MaxDD drift vs BC exceeds `2pp`; observed PPO-selected drift was `{handoff['nearest_known_gap']['wf_2022_ppo_selected_max_dd_delta_vs_bc']:.6f}pp`.",
    '- Next retraining run must add wf_2020 coverage before any actor retention, then rerun fixed-split + full walk-forward + dashboard replay export.',
]
(CARD / 'conclusion.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')

print(json.dumps(summary_out, indent=2, sort_keys=True))


{
  "created_at": "2026-06-13T20:16:36.217469+00:00",
  "decision": "run_full_walkforward_cap10_joint_contract_bc_checkpoint_training",
  "do_not_repeat": [
    "do_not_promote_any_replayable_model_without_complete_wf_2020_wf_2022_wf_2023_2024_metrics",
    "do_not_continue_static_heuristic_or_rolling_selector_branch_without_model_replay_or_bc_candidate_paths",
    "do_not_lengthen_ppo_when_wf_2022_dd_delta_vs_bc_exceeds_2pp",
    "do_not_retry_rebound_weight_only_bc_without_target_enrichment_or_sampling_change"
  ],
  "experiment": "retraining_gate_gap_handoff_v1",
  "nearest_known_gap": {
    "wf_2022_ppo_selected_max_dd_delta_vs_bc": 18.820132,
    "wf_2022_ppo_selected_relative_delta_vs_bc": 5.351209,
    "wf_2022_selected_max_dd_gap_to_15pct": 3.455088,
    "wf_2022_selected_max_dd_gap_to_20pct": 0.0,
    "wf_2023_2024_selected_gate": true
  },
  "next_training_allowed": true,
  "promotion_allowed": false,
  "required_next_run_controls": [
    "add_wf_2020_coverage_before_actor_re

## 2026-06-14 strict gate refresh

This refresh adds the current strict retention gate, qlib-aligned SOXLBacktest buy/sell reference, walk-forward weighted objective screen, and dashboard feature contract to the retraining handoff. It keeps Vercel out of scope for this iteration and treats the local dashboard/server bundle as the only UI target.

In [5]:
# ## 2026-06-14 strict gate refresh: load newer source cards and build the frontier gap table
ADDITIONAL_SOURCES = {
    'soxlbacktest_qlib_summary': 'runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/summary.json',
    'soxlbacktest_qlib_gate_csv': 'runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/candidate_gate_summary.csv',
    'walkforward_weighted_selection': 'runtime/model_selection/card/walkforward_weighted_objective_v1/selection.json',
    'walkforward_weighted_scores': 'runtime/model_selection/card/walkforward_weighted_objective_v1/walkforward_weighted_scores.csv',
    'reference_model_score': 'runtime/model_selection/card/soxlbacktest_reference_model_score_v1/summary.json',
    'reference_model_score_csv': 'runtime/model_selection/card/soxlbacktest_reference_model_score_v1/candidate_score_summary.csv',
    'state_overlay_summary': 'runtime/model_selection/card/state_conditioned_overlay_gap_to_20_v1/summary.json',
    'state_overlay_csv': 'runtime/model_selection/card/state_conditioned_overlay_gap_to_20_v1/overlay_gate_summary.csv',
}

soxl_qlib = load_json(ADDITIONAL_SOURCES['soxlbacktest_qlib_summary'])
soxl_gate = pd.read_csv(ROOT / ADDITIONAL_SOURCES['soxlbacktest_qlib_gate_csv'])
weighted_selection = load_json(ADDITIONAL_SOURCES['walkforward_weighted_selection'])
weighted_scores = pd.read_csv(ROOT / ADDITIONAL_SOURCES['walkforward_weighted_scores'])
reference_model_score = load_json(ADDITIONAL_SOURCES['reference_model_score'])
model_score_rows = pd.read_csv(ROOT / ADDITIONAL_SOURCES['reference_model_score_csv'])
state_overlay = load_json(ADDITIONAL_SOURCES['state_overlay_summary'])
overlay_rows = pd.read_csv(ROOT / ADDITIONAL_SOURCES['state_overlay_csv'])

thresholds = soxl_qlib['thresholds']
max_wf_dd = float(thresholds['max_walkforward_drawdown_pct'])
required_best = float(thresholds['required_best_return_pct'])
required_soxl = float(thresholds['required_soxlbacktest_return_pct'])

frontier_records = []

def add_frontier_record(**kwargs):
    record = {
        'candidate': kwargs.get('candidate'),
        'source': kwargs.get('source'),
        'role': kwargs.get('role', ''),
        'ytd_return_pct': kwargs.get('ytd_return_pct'),
        'worst_wf_max_drawdown_pct': kwargs.get('worst_wf_max_drawdown_pct'),
        'dd_gap_to_20_pct': kwargs.get('dd_gap_to_20_pct'),
        'return_gap_to_soxlbacktest_pct': kwargs.get('return_gap_to_soxlbacktest_pct'),
        'return_gap_to_current_best_floor_pct': kwargs.get('return_gap_to_current_best_floor_pct'),
        'min_wf_return_pct': kwargs.get('min_wf_return_pct'),
        'mean_wf_return_pct': kwargs.get('mean_wf_return_pct'),
        'min_relative_return_pp': kwargs.get('min_relative_return_pp'),
        'weighted_objective_loss': kwargs.get('weighted_objective_loss'),
        'dominant_failure_type': kwargs.get('dominant_failure_type'),
        'worst_window': kwargs.get('worst_window'),
        'retention_gate_pass': bool(kwargs.get('retention_gate_pass', False)),
        'failures': kwargs.get('failures', ''),
        'next_action': kwargs.get('next_action', ''),
    }
    frontier_records.append(record)

for _, row in soxl_gate.iterrows():
    ytd = as_float(row.get('ytd_return_pct'))
    wf_dd = as_float(row.get('worst_wf_max_drawdown_pct'))
    add_frontier_record(
        candidate=row['candidate'],
        source='soxlbacktest_qlib_walkforward_baseline_v1',
        role=row.get('role', ''),
        ytd_return_pct=ytd,
        worst_wf_max_drawdown_pct=wf_dd,
        dd_gap_to_20_pct=max(0.0, wf_dd - max_wf_dd),
        return_gap_to_soxlbacktest_pct=required_soxl - ytd,
        return_gap_to_current_best_floor_pct=required_best - ytd,
        min_wf_return_pct=row.get('min_wf_return_pct'),
        mean_wf_return_pct=row.get('mean_wf_return_pct'),
        retention_gate_pass=bool(row.get('retention_gate_pass')),
        failures=row.get('failures', ''),
        next_action='use_as_buy_sell_reference' if row['candidate'] == soxl_qlib['selected_buy_sell_reference'] else 'use_as_risk_floor_reference_only',
    )

retention_candidates = pd.concat([closest_by_wf_dd, closest_by_ytd]).drop_duplicates(subset=['profile'])
for _, row in retention_candidates.iterrows():
    ytd = as_float(row.get('ytd_return_pct'))
    wf_dd = as_float(row.get('worst_wf_max_drawdown_pct'), default=float('nan'))
    add_frontier_record(
        candidate=row['profile'],
        source='model_retention_walkforward_gate_v1',
        role=row.get('status', ''),
        ytd_return_pct=ytd,
        worst_wf_max_drawdown_pct=wf_dd,
        dd_gap_to_20_pct=None if pd.isna(wf_dd) else max(0.0, wf_dd - max_wf_dd),
        return_gap_to_soxlbacktest_pct=required_soxl - ytd,
        return_gap_to_current_best_floor_pct=required_best - ytd,
        retention_gate_pass=bool(row.get('gate_pass')),
        failures=row.get('failures', ''),
        next_action='retain_as_evidence_only_until_complete_gate_pass',
    )

for _, row in weighted_scores.head(8).iterrows():
    wf_dd = as_float(row.get('max_max_dd'))
    add_frontier_record(
        candidate=row['candidate'],
        source='walkforward_weighted_objective_v1',
        role='diagnostic_candidate',
        worst_wf_max_drawdown_pct=wf_dd,
        dd_gap_to_20_pct=max(0.0, wf_dd - max_wf_dd),
        min_relative_return_pp=row.get('min_relative_return'),
        weighted_objective_loss=row.get('weighted_objective_loss'),
        dominant_failure_type=row.get('dominant_failure_type'),
        worst_window=row.get('worst_window'),
        retention_gate_pass=bool(row.get('full_gate')),
        failures='full_gate_false' if not bool(row.get('full_gate')) else '',
        next_action='use_as_walkforward_weighted_training_reference' if row['candidate'] == weighted_selection['selected_diagnostic_candidate'] else 'diagnostic_only',
    )

for _, row in overlay_rows.head(6).iterrows():
    ytd = as_float(row.get('ytd_return_pct'))
    wf_dd = as_float(row.get('worst_wf_max_drawdown_pct'))
    add_frontier_record(
        candidate=row['variant'],
        source='state_conditioned_overlay_gap_to_20_v1',
        role='overlay_probe',
        ytd_return_pct=ytd,
        worst_wf_max_drawdown_pct=wf_dd,
        dd_gap_to_20_pct=max(0.0, wf_dd - max_wf_dd),
        return_gap_to_soxlbacktest_pct=required_soxl - ytd,
        return_gap_to_current_best_floor_pct=required_best - ytd,
        min_wf_return_pct=row.get('min_wf_return_pct'),
        mean_wf_return_pct=row.get('mean_wf_return_pct'),
        retention_gate_pass=bool(row.get('retention_gate_pass')),
        failures=row.get('failures', ''),
        next_action='reject_global_position_cap_pattern_if_return_floor_fails',
    )

frontier_gap = pd.DataFrame(frontier_records)
frontier_gap['sort_dd_gap'] = frontier_gap['dd_gap_to_20_pct'].fillna(999.0)
frontier_gap['sort_return_gap'] = frontier_gap['return_gap_to_current_best_floor_pct'].fillna(999.0)
frontier_gap = frontier_gap.sort_values(['retention_gate_pass', 'sort_dd_gap', 'sort_return_gap'], ascending=[False, True, True]).drop(columns=['sort_dd_gap', 'sort_return_gap'])
frontier_gap.to_csv(CARD / 'strict_gate_frontier_gap.csv', index=False)

alignment_cols = [
    'profile', 'status', 'has_ytd_daily', 'wf_windows_present', 'missing_wf_windows',
    'ytd_return_pct', 'worst_wf_max_drawdown_pct', 'mean_baseline_position_mae',
    'mean_baseline_event_match_rate', 'mean_risk_floor_position_mae', 'complete_wf',
    'wf_drawdown_pass', 'ytd_soxlbacktest_return_pass', 'ytd_current_best_alignment_pass',
    'retention_gate_pass', 'selection_score', 'failures',
]
buy_sell_alignment = model_score_rows[[col for col in alignment_cols if col in model_score_rows.columns]].copy()
buy_sell_alignment = buy_sell_alignment.sort_values(['retention_gate_pass', 'selection_score'], ascending=[False, False])
buy_sell_alignment.to_csv(CARD / 'buy_sell_reference_alignment.csv', index=False)

print('frontier_gap_rows', len(frontier_gap))
print(frontier_gap.head(12).to_string(index=False))
print('buy_sell_alignment_top', buy_sell_alignment.head(5).to_dict(orient='records'))


frontier_gap_rows 20
                                 candidate                                    source                              role  ytd_return_pct  worst_wf_max_drawdown_pct  dd_gap_to_20_pct  return_gap_to_soxlbacktest_pct  return_gap_to_current_best_floor_pct  min_wf_return_pct  mean_wf_return_pct  min_relative_return_pp  weighted_objective_loss dominant_failure_type worst_window  retention_gate_pass                                                                                                    failures                                              next_action
                    wf_pass_ytd18_15_15_02 soxlbacktest_qlib_walkforward_baseline_v1   wf_drawdown_pass_best_ytd_probe       18.308910                  18.514585          0.000000                      113.769068                            194.834552         -16.044551            5.872611                     NaN                      NaN                  None         None                False                           

In [6]:
# ## 2026-06-14 strict gate refresh: enrich the handoff contract and conclusion with qlib baseline + dashboard feature scope
base_handoff = json.loads((CARD / 'training_objective_handoff.json').read_text(encoding='utf-8'))
selected_weighted_row = weighted_scores[weighted_scores['candidate'].eq(weighted_selection['selected_diagnostic_candidate'])].iloc[0].to_dict()
legacy_default = soxl_qlib['legacy_default']
risk_floor = soxl_qlib[soxl_qlib['selected_risk_floor']]
reference_profile = summary['reference_profile']
reference_row_count = int(retention_rows['profile'].eq(reference_profile).sum())
replayable_profile_count = int(len(profile_rows))
strict_retention_current_scope = {
    'replayable_profile_rows': replayable_profile_count,
    'reference_baseline_rows': reference_row_count,
    'reference_profile': reference_profile,
    'passing_replayable_profiles': list(summary.get('passing_profiles', [])),
    'protected_keep_actor_exceptions': summary.get('registry_keep_actor_protected_exception_count'),
    'protected_keep_actor_strict_violations': summary.get('registry_keep_actor_strict_violations', []),
}

def unique_list(values):
    seen = set()
    out = []
    for value in values:
        if value not in seen:
            seen.add(value)
            out.append(value)
    return out

sidecars = unique_list(base_handoff.get('sidecars', []) + [
    'strict_gate_frontier_gap.csv',
    'buy_sell_reference_alignment.csv',
])

do_not_repeat = unique_list(base_handoff.get('do_not_repeat', []) + [
    'do_not_treat_soxlbacktest_low_risk_caps_as_promotable_model',
    'do_not_treat_reference_baseline_as_runnable_inference_profile',
    'do_not_consider_vercel_deploy_status_as_dashboard_fix_scope',
    'do_not_ship_dashboard_replay_without_model_specific_feature_snapshot',
])

required_controls = unique_list(base_handoff.get('required_next_run_controls', []) + [
    'score_candidate_buy_sell_events_against_soxlbacktest_legacy_default_reference',
    'export_feature_packs_feature_columns_and_feature_snapshot_for_dashboard_replay',
    'export_feature_action_summary_buy_sell_mean_deltas_for_dashboard_v2',
    'refresh_dashboard_model_replays_then_build_dashboard_bundle_after_training',
])

updated_handoff = dict(base_handoff)
updated_handoff.update({
    'updated_at': datetime.now(timezone.utc).isoformat(),
    'decision': 'run_full_walkforward_cap10_joint_contract_bc_checkpoint_training_with_qlib_buy_sell_reference',
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'selected_training_source': 'cap10_joint_contract_checkpoint_drift_selector_v1_plus_walkforward_weighted_reference',
    'soxlbacktest_qlib_reference': {
        'card': 'soxlbacktest_qlib_walkforward_baseline_v1',
        'selected_buy_sell_reference': soxl_qlib['selected_buy_sell_reference'],
        'legacy_default_ytd_return_pct': legacy_default['ytd_return_pct'],
        'legacy_default_worst_wf_max_drawdown_pct': legacy_default['worst_wf_max_drawdown_pct'],
        'selected_risk_floor': soxl_qlib['selected_risk_floor'],
        'risk_floor_ytd_return_pct': risk_floor['ytd_return_pct'],
        'risk_floor_worst_wf_max_drawdown_pct': risk_floor['worst_wf_max_drawdown_pct'],
        'interpretation': 'legacy_default is the buy/sell timing reference; low-risk variants show drawdown can pass only by destroying return',
    },
    'walkforward_weighted_reference': {
        'card': 'walkforward_weighted_objective_v1',
        'selected_diagnostic_candidate': weighted_selection['selected_diagnostic_candidate'],
        'next_mechanism': weighted_selection['next_mechanism'],
        'selected_weighted_objective_loss': weighted_selection['selected_weighted_objective_loss'],
        'selected_max_wf_drawdown_pct': selected_weighted_row['max_max_dd'],
        'selected_min_relative_return_pp': selected_weighted_row['min_relative_return'],
        'dominant_failure_type': selected_weighted_row['dominant_failure_type'],
        'target': weighted_selection['target'],
        'strict_vs_walkforward': weighted_selection['strict_vs_walkforward'],
    },
    'strict_retention_current_scope': strict_retention_current_scope,
    'dashboard_contract': {
        'scope': 'local_dashboard_and_dashboard_server_only',
        'vercel': 'excluded_per_user_request',
        'auto_refresh_entrypoints': [
            'production/soxl_daily_scheduler.py',
            'production/run_dashboard_refresh.sh',
            'scripts/build_dashboard_bundle.py',
        ],
        'live_inference_entrypoint': 'scripts/dashboard_server.py',
        'feature_requirements': [
            'each model exposes its own feature_packs and feature_columns',
            'rerun/API rows preserve feature_snapshot by date',
            'dashboard v2 renders BUY/SELL feature mean deltas and recent trade snapshots',
            'reference baselines remain non-runnable and must not be offered as inference models',
        ],
    },
    'do_not_repeat': do_not_repeat,
    'required_next_run_controls': required_controls,
    'sidecars': sidecars,
})

(CARD / 'training_objective_handoff.json').write_text(json.dumps(updated_handoff, indent=2, sort_keys=True) + '\n', encoding='utf-8')

summary_out = {
    'experiment': updated_handoff['experiment'],
    'created_at': base_handoff.get('created_at'),
    'updated_at': updated_handoff['updated_at'],
    'decision': updated_handoff['decision'],
    'promotion_allowed': updated_handoff['promotion_allowed'],
    'actor_retention_allowed': updated_handoff['actor_retention_allowed'],
    'next_training_allowed': updated_handoff['next_training_allowed'],
    'retention_gate': updated_handoff['retention_gate'],
    'start_from': updated_handoff['start_from'],
    'nearest_known_gap': updated_handoff['nearest_known_gap'],
    'selected_training_source': updated_handoff['selected_training_source'],
    'soxlbacktest_qlib_reference': updated_handoff['soxlbacktest_qlib_reference'],
    'walkforward_weighted_reference': updated_handoff['walkforward_weighted_reference'],
    'strict_retention_current_scope': updated_handoff['strict_retention_current_scope'],
    'dashboard_contract': updated_handoff['dashboard_contract'],
    'do_not_repeat': updated_handoff['do_not_repeat'],
    'required_next_run_controls': updated_handoff['required_next_run_controls'],
    'sidecars': updated_handoff['sidecars'],
}
(CARD / 'summary.json').write_text(json.dumps(summary_out, indent=2, sort_keys=True) + '\n', encoding='utf-8')

source_cards_path = CARD / 'source_card_decisions.csv'
if source_cards_path.exists():
    source_cards_current = pd.read_csv(source_cards_path)
    mask = source_cards_current['source'].eq('model_retention_walkforward_gate_v1')
    source_cards_current.loc[mask, 'key_evidence'] = (
        f"strict retention CSV has {replayable_profile_count} replayable model rows plus "
        f"{reference_row_count} reference baseline row; no replayable model passes. "
        f"Protected keep_actor strict violations: {summary.get('registry_keep_actor_strict_violation_count', 0)}"
    )
    source_cards_current.to_csv(source_cards_path, index=False)

conclusion_lines = [
    '# Retraining Gate Gap Handoff V1',
    '',
    f"- Decision: `{updated_handoff['decision']}`.",
    '- Promotion: blocked; this card does not retain or promote a new actor.',
    f"- Strict retention scope: {replayable_profile_count} replayable model rows plus {reference_row_count} SOXLBacktest reference row; no replayable model passes complete WF + MaxDD/return alignment.",
    f"- SOXLBacktest qlib reference: `{soxl_qlib['selected_buy_sell_reference']}` keeps the buy/sell timing baseline with YTD `{legacy_default['ytd_return_pct']:.6f}%`, but worst WF MaxDD is `{legacy_default['worst_wf_max_drawdown_pct']:.6f}%` and is not promotable.",
    f"- SOXLBacktest risk-floor probe: `{soxl_qlib['selected_risk_floor']}` passes the <20% WF drawdown screen with MaxDD `{risk_floor['worst_wf_max_drawdown_pct']:.6f}%`, but YTD is only `{risk_floor['ytd_return_pct']:.6f}%`; use it as a floor reference, not a model.",
    f"- Walk-forward weighted diagnostic: `{weighted_selection['selected_diagnostic_candidate']}` is closest by weighted loss, but still fails full gate with max WF MaxDD `{selected_weighted_row['max_max_dd']:.6f}%`; next mechanism is `{weighted_selection['next_mechanism']}`.",
    f"- Best available training start remains cap10 joint-contract BC/checkpoint path. wf_2022 selected MaxDD `{updated_handoff['start_from']['wf_2022_selected_max_dd']:.6f}%`, rel `{updated_handoff['start_from']['wf_2022_selected_relative_return']:.6f}pp`; wf_2023_2024 selected MaxDD `{updated_handoff['start_from']['wf_2023_2024_selected_max_dd']:.6f}%`, rel `{updated_handoff['start_from']['wf_2023_2024_selected_relative_return']:.6f}pp`.",
    f"- Critical stop rule: do not lengthen PPO when wf_2022 MaxDD drift vs BC exceeds `2pp`; observed PPO-selected drift was `{updated_handoff['nearest_known_gap']['wf_2022_ppo_selected_max_dd_delta_vs_bc']:.6f}pp`.",
    '- Dashboard scope: local dashboard/server only; Vercel is excluded. The next retained candidate must export per-model feature columns, feature snapshots, BUY/SELL feature summaries, position, action, and price fields before it can be exposed as runnable in v2.',
    '- Next action: run full walk-forward training with wf_2020 coverage, qlib SOXLBacktest buy/sell reference scoring, and dashboard replay export; keep actor retention blocked until all gates pass.',
]
(CARD / 'conclusion.md').write_text('\n'.join(conclusion_lines) + '\n', encoding='utf-8')

print(json.dumps(summary_out, indent=2, sort_keys=True))


{
  "actor_retention_allowed": false,
  "created_at": "2026-06-13T20:16:36.217469+00:00",
  "dashboard_contract": {
    "auto_refresh_entrypoints": [
      "production/soxl_daily_scheduler.py",
      "production/run_dashboard_refresh.sh",
      "scripts/build_dashboard_bundle.py"
    ],
    "feature_requirements": [
      "each model exposes its own feature_packs and feature_columns",
      "rerun/API rows preserve feature_snapshot by date",
      "dashboard v2 renders BUY/SELL feature mean deltas and recent trade snapshots",
      "reference baselines remain non-runnable and must not be offered as inference models"
    ],
    "live_inference_entrypoint": "scripts/dashboard_server.py",
    "scope": "local_dashboard_and_dashboard_server_only",
    "vercel": "excluded_per_user_request"
  },
  "decision": "run_full_walkforward_cap10_joint_contract_bc_checkpoint_training_with_qlib_buy_sell_reference",
  "do_not_repeat": [
    "do_not_promote_any_replayable_model_without_complete_wf_2020_wf

In [7]:
# ## 2026-06-14 strict gate refresh: executable conclusion cell
from IPython.display import Markdown, display

final_summary = json.loads((CARD / 'summary.json').read_text(encoding='utf-8'))
message = f"""
## Conclusion

Decision: `{final_summary['decision']}`. No actor is promoted or retained from this pass.

The next training branch should start from `{final_summary['selected_training_source']}`, score buy/sell timing against SOXLBacktest `{final_summary['soxlbacktest_qlib_reference']['selected_buy_sell_reference']}`, and use `{final_summary['walkforward_weighted_reference']['selected_diagnostic_candidate']}` only as the closest walk-forward-weighted diagnostic reference. The low-risk SOXLBacktest probes show how to pass drawdown by sacrificing return, so they are risk-floor references only.

Dashboard handoff is local-only: refresh model replays, run `scripts/build_dashboard_bundle.py`, and expose model-specific `feature_packs`, `feature_columns`, `feature_snapshot`, and BUY/SELL feature summaries. Vercel is explicitly outside this iteration.
""".strip()
display(Markdown(message))


## Conclusion

Decision: `run_full_walkforward_cap10_joint_contract_bc_checkpoint_training_with_qlib_buy_sell_reference`. No actor is promoted or retained from this pass.

The next training branch should start from `cap10_joint_contract_checkpoint_drift_selector_v1_plus_walkforward_weighted_reference`, score buy/sell timing against SOXLBacktest `legacy_default`, and use `recovery_reentry_conditional_vol_target_v1` only as the closest walk-forward-weighted diagnostic reference. The low-risk SOXLBacktest probes show how to pass drawdown by sacrificing return, so they are risk-floor references only.

Dashboard handoff is local-only: refresh model replays, run `scripts/build_dashboard_bundle.py`, and expose model-specific `feature_packs`, `feature_columns`, `feature_snapshot`, and BUY/SELL feature summaries. Vercel is explicitly outside this iteration.